# Importing

In [ ]:
import io
import os
import pickle
import time
import warnings
import librosa
import numpy as np
import pandas as pd
import tensorflow as tf
!pip install azure-storage-blob
!pip install seaborn python-dotenv librosa scikit-learn
from azure.storage.blob import BlobServiceClient
from dotenv import load_dotenv
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau,
)
import tensorflow.keras.layers as L
from audio_pipeline import extract_features_from_audio_array
from config import (
    ASSETS_DIR,
    ENCODER_PATH,
    MODEL_PATH,
    MONO,
    SCALER_PATH,
    TARGET_SR,
)
from manifest_builder import (
    add_augmented_rows,
    assign_splits,
    build_manifest,
    save_manifest,
    validate_manifest,
)
from speaker_splitter import add_speaker_column, speaker_split

warnings.filterwarnings("ignore")
load_dotenv()


# Integration with Azure

In [ ]:
from azure.storage.blob import BlobServiceClient
import pandas as pd
import os
from dotenv import load_dotenv
import numpy as np
import io
import librosa


load_dotenv()

CONTAINER_NAME = "wav-files"

connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")

if not connection_string:
    raise ValueError("AZURE_STORAGE_CONNECTION_STRING not set")

blob_service = BlobServiceClient.from_connection_string(connection_string)
container_client = blob_service.get_container_client(CONTAINER_NAME)


def get_df_from_blob_dataset(prefix):
    emotions = []
    file_paths = []

    for blob in container_client.list_blobs(name_starts_with=prefix):
        if blob.name.endswith(".wav"):
            filename = blob.name.split("/")[-1]
            parts = filename.split("-")

            if len(parts) >= 3:
                emotion = parts[2].lower()
            else:
                emotion = "unknown"

            emotions.append(emotion)
            file_paths.append(blob.name)

    return pd.DataFrame({
        "Path": file_paths,
        "Emotions": emotions
    })


def load_audio_from_blob(blob_service, blob_path: str) -> tuple[np.ndarray, int]:
    clean_path = blob_path.replace("\\", "/")
    blob_client = blob_service.get_blob_client(container=CONTAINER_NAME, blob=clean_path)
    audio_file = io.BytesIO(blob_client.download_blob().readall())
    data, sr = librosa.load(audio_file, sr=TARGET_SR, mono=MONO)
    return data.astype(np.float32), sr



ravdess_df = get_df_from_blob_dataset("RAVDESS/")
crema_df   = get_df_from_blob_dataset("CREMAD/")
tess_df    = get_df_from_blob_dataset("TESS/")
savee_df   = get_df_from_blob_dataset("SAVEE/")

data_path = pd.concat(
    [ravdess_df, crema_df, tess_df, savee_df],
    axis=0
).reset_index(drop=True)

data_path.to_csv("data_path.csv", index=False)

print("Total files:", len(data_path))
print(data_path.head())
print("\nEmotion distribution:")
print(data_path["Emotions"].value_counts())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure()
sns.countplot(
    x=data_path["Emotions"],  
    color="purple"             
)
plt.title('Count of Emotions', size=16)
plt.xlabel('Emotion', size=12)
plt.ylabel('Count', size=12)

sns.despine(top=True, right=True, left=False, bottom=False)

plt.show()

# Splitting the data before augmentation

In [ ]:
import os
import io
import time
import warnings
import librosa
import numpy as np
import pandas as pd
from azure.storage.blob import BlobServiceClient
from dotenv import load_dotenv
from speaker_splitter import add_speaker_column, speaker_split
from config import TARGET_SR, MONO
from audio_pipeline import extract_features_from_audio_array

warnings.filterwarnings("ignore")

df = pd.read_csv("data_path.csv")


df = add_speaker_column(df)
train_df, test_df = speaker_split(df)

print(f"Original files total: {len(df)}")
print(f"Train original files: {len(train_df)}")
print(f"Test original files:  {len(test_df)}")

# Load audio from Azure 

In [ ]:
"""def load_audio_from_blob(path: str):
    clean_path = path.replace("\\", "/")

    blob_client = blob_service.get_blob_client(
        container=CONTAINER_NAME,
        blob=clean_path
    )

    audio_bytes = blob_client.download_blob().readall()
    audio_file = io.BytesIO(audio_bytes)

    data, sr = librosa.load(audio_file, sr=TARGET_SR, mono=MONO)

    return data.astype(np.float32), sr

# Data Augmentation

In [ ]:

def noise(data, sr=None):
    noise_amp = 0.035 * np.random.uniform() * np.max(np.abs(data))
    return data + noise_amp * np.random.normal(size=data.shape[0])


def stretch(data, sr=None, rate=0.8):
    return librosa.effects.time_stretch(y=data, rate=rate)


def shift(data, sr=None):
    shift_range = int(np.random.uniform(-5, 5) * 1000)
    return np.roll(data, shift_range)


def pitch(data, sr, n_steps=0.7):
    return librosa.effects.pitch_shift(y=data, sr=sr, n_steps=n_steps)


AUGMENTATIONS = {
    "noise":   lambda d, sr: noise(d, sr),
    "stretch": lambda d, sr: stretch(d, sr),
    "shift":   lambda d, sr: shift(d, sr),
    "pitch":   lambda d, sr: pitch(d, sr),
}

# Build Feature Extraction

In [ ]:
def build_feature_dataframe(
    input_df: pd.DataFrame,
    blob_service: BlobServiceClient,
    augment: bool,
    split_name: str,
) -> tuple[pd.DataFrame, list[dict]]:
    X, Y, source_paths, is_augmented_flags, aug_types = [], [], [], [], []
    aug_records: list[dict] = []

    print(f"\nStarting {split_name}: {len(input_df)} original files")
    t0 = time.time()

    for i, (_, row) in enumerate(input_df.iterrows(), 1):
        path    = row["Path"]
        emotion = row["Emotions"]

        try:
            data, sr = load_audio_from_blob(blob_service, path)
        except Exception as e:
            print(f"  [ERROR] {path}: {e}")
            continue

        feat = extract_features_from_audio_array(data, sr)
        X.append(feat); Y.append(emotion)
        source_paths.append(path); is_augmented_flags.append(False)
        aug_types.append("original")

        if augment: #  if it's train file - do augmentation
            for aug_name, aug_fn in AUGMENTATIONS.items():
                try:
                    aug_data = aug_fn(data, sr)
                    aug_feat = extract_features_from_audio_array(aug_data, sr)
                    X.append(aug_feat); Y.append(emotion)
                    source_paths.append(path); is_augmented_flags.append(True)
                    aug_types.append(aug_name)
                    aug_records.append({
                        "source_original_path": path,
                        "augmentation_type": aug_name,
                    })
                except Exception:
                    pass

        if i % 100 == 0:
            print(f"  {split_name}: {i}/{len(input_df)} files ({(time.time()-t0)/60:.1f} min)")

    print(f"  Finished in {(time.time()-t0)/60:.1f} min ({len(X)} total samples)")

    features_df = pd.DataFrame(X)
    features_df["Labels"]               = Y
    features_df["source_original_path"] = source_paths
    features_df["is_augmented"]         = is_augmented_flags
    features_df["augmentation_type"]    = aug_types
    features_df["split"]                = split_name

    return features_df, aug_records


# Speaker Split & Manifest Creation

In [ ]:
# 1. Speaker split
data_path = add_speaker_column(data_path)
train_df, test_df = speaker_split(data_path, test_size=0.2, random_state=42)

# 2.Build manifest
manifest = build_manifest(data_path)
manifest = assign_splits(manifest, train_df, test_df)

# 3. Extract features 
train_features_df, aug_records = build_feature_dataframe(
    train_df, blob_service, augment=True, split_name="train"
)
test_features_df, _ = build_feature_dataframe(
    test_df, blob_service, augment=False, split_name="test"
)

# 4. Save feature datasets
train_features_df.to_csv("train_features_ready_for_model.csv", index=False)
test_features_df.to_csv("test_features_ready_for_model.csv", index=False)

# 5. Finalize and save the manifest
manifest = add_augmented_rows(manifest, aug_records)
validate_manifest(manifest)
save_manifest(manifest)


# Data Preparation (Scaling & Encoding)

In [ ]:
train_df2 = pd.read_csv("train_features_ready_for_model.csv").fillna(0)
test_df2  = pd.read_csv("test_features_ready_for_model.csv").fillna(0)

meta_cols  = ["Labels", "source_original_path", "is_augmented", "augmentation_type", "split"]
feat_cols  = [c for c in train_df2.columns if c not in meta_cols]

X_train = train_df2[feat_cols].values
X_test  = test_df2[feat_cols].values
y_train_raw = train_df2["Labels"].values
y_test_raw  = test_df2["Labels"].values

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
y_train = encoder.fit_transform(y_train_raw.reshape(-1, 1))
y_test  = encoder.transform(y_test_raw.reshape(-1, 1))

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

x_traincnn = np.expand_dims(X_train, axis=2)
x_testcnn  = np.expand_dims(X_test,  axis=2)

print(f"Train CNN shape: {x_traincnn.shape}  Test CNN shape: {x_testcnn.shape}")


# CNN Arcticeture

In [ ]:
model = tf.keras.Sequential([
    L.Conv1D(512, 5, padding="same", activation="relu", input_shape=(x_traincnn.shape[1], 1)),
    L.BatchNormalization(),
    L.MaxPool1D(5, strides=2, padding="same"),

    L.Conv1D(512, 5, padding="same", activation="relu"),
    L.BatchNormalization(),
    L.MaxPool1D(5, strides=2, padding="same"),
    L.Dropout(0.2),

    L.Conv1D(256, 5, padding="same", activation="relu"),
    L.BatchNormalization(),
    L.MaxPool1D(5, strides=2, padding="same"),

    L.Conv1D(256, 3, padding="same", activation="relu"),
    L.BatchNormalization(),
    L.MaxPool1D(5, strides=2, padding="same"),
    L.Dropout(0.2),

    L.Conv1D(128, 3, padding="same", activation="relu"),
    L.BatchNormalization(),
    L.MaxPool1D(3, strides=2, padding="same"),
    L.Dropout(0.2),

    L.Flatten(),
    L.Dense(512, activation="relu"),
    L.BatchNormalization(),
    L.Dropout(0.2),

    L.Dense(y_train.shape[1], activation="softmax"),
])

model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

callbacks = [
    ModelCheckpoint(str(MODEL_PATH), monitor="val_accuracy", save_best_only=True, verbose=1), #checkpoint- save the best model and delete the rest
    EarlyStopping(monitor="val_accuracy", mode="max", patience=10, restore_best_weights=True, verbose=1), #Early Stopping: Stops training if the model doesn't improve for 10 epochs

    ReduceLROnPlateau(monitor="val_accuracy", patience=5, factor=0.5, min_lr=1e-5, verbose=1),#if the model doesn't improve for 5 epochs, reduce the steps by 50% (gradient descent)
]



# CNN Model

In [ ]:
import matplotlib.pyplot as plt


# --- The actual training step ---
history = model.fit(
    x_traincnn, y_train,
    epochs=50,
    validation_data=(x_testcnn, y_test),
    batch_size=64,
    callbacks=callbacks,
)

print(" Model has finished training.")


print("Plotting training history...")
epochs = range(len(history.history['accuracy']))
fig, ax = plt.subplots(1, 2, figsize=(20, 6))

# Plotting Loss
ax[0].plot(epochs, history.history['loss'], label='Training Loss', color='blue')
ax[0].plot(epochs, history.history['val_loss'], label='Validation Loss', color='red')
ax[0].set_title('Training & Validation Loss')
ax[0].legend()
ax[0].set_xlabel("Epochs")

# Plotting Accuracy
ax[1].plot(epochs, history.history['accuracy'], label='Training Accuracy', color='blue')
ax[1].plot(epochs, history.history['val_accuracy'], label='Validation Accuracy', color='red')
ax[1].set_title('Training & Validation Accuracy')
ax[1].legend()
ax[1].set_xlabel("Epochs")

plt.show()

In [ ]:
# predicting on test data.
pred_test0 = model.predict(x_testcnn)
y_pred0 = encoder.inverse_transform(pred_test0)
y_test0 = encoder.inverse_transform(y_test)

# Check for random predictions
df0 = pd.DataFrame(columns=['Predicted Labels', 'Actual Labels'])
df0['Predicted Labels'] = y_pred0.flatten()
df0['Actual Labels'] = y_test0.flatten()

df0.head(10)

# Evaluation

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt


pred_test = model.predict(x_testcnn)

y_pred_labels = encoder.inverse_transform(pred_test)
y_test_labels = encoder.inverse_transform(y_test)

cm = confusion_matrix(y_test_labels, y_pred_labels)

labels = encoder.categories_[0]

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', 
            xticklabels=labels, yticklabels=labels)

plt.title('Confusion Matrix - Audio Emotion Recognition', size=20)
plt.xlabel('Predicted Labels (What the model thought)', size=14)
plt.ylabel('Actual Labels (The Truth)', size=14)
plt.show()

print("\n--- Detailed Classification Report ---")
print(classification_report(y_test_labels, y_pred_labels))

# Saving the model

In [ ]:
import os
import numpy as np
import librosa
import pickle
from tensorflow.keras.models import load_model


FOLDER_NAME = 'model_assets'

if not os.path.exists(FOLDER_NAME):
    os.makedirs(FOLDER_NAME)
    print(f"Directory created: {FOLDER_NAME}")

model_path = os.path.join(FOLDER_NAME, 'final_emotion_model.keras')
model.save(model_path)
print(f"Model saved to: {model_path}")


with open(os.path.join(FOLDER_NAME, 'scaler.pickle'), 'wb') as f:
    pickle.dump(scaler, f)

with open(os.path.join(FOLDER_NAME, 'encoder.pickle'), 'wb') as f:
    pickle.dump(encoder, f)

print(f"Preprocessing tools saved to: {FOLDER_NAME}/")


def extract_features_single(path):
    return extract_features_from_file(path)

def predict_emotion(audio_path):
    # Step A: Extract features
    feat = extract_features_single(audio_path)
    
    # Step B: Scale features using the scaler we saved earlier
    feat = scaler.transform(feat.reshape(1, -1))
    
    # Step C: Reshape for CNN (Add the 3rd dimension)
    feat = np.expand_dims(feat, axis=2)
    
    # Step D: Predict probabilities
    predictions = model.predict(feat, verbose=0)
    
    # Step E: Get the emotion label and confidence
    emotion = encoder.inverse_transform(predictions)
    confidence = np.max(predictions)
    
    return emotion[0][0], confidence

print("\nSystem ready for new predictions using assets from the folder!")